---
title: "Spark SQL Job Data Analysis"
author: "Zhengyu Zhou"
format: html
embed-resources: true
date: "2025-10-18"
date-format: long
execute:
  echo: true
---

In [24]:
import pandas as pd

# read .xlsx and convert to .csv
excel_path = "data/job_postings.xlsx"   
csv_path   = "data/job_postings.csv"

# csv file
df = pd.read_excel(excel_path, engine="openpyxl")
df.to_csv(csv_path, index=False)

print("Converted", csv_path)
print("len:", len(df), "columns:", len(df.columns))

KeyboardInterrupt: 

In [ ]:
from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", "\"").csv("data/job_postings.csv")

# 3. Register the DataFrame as a temporary SQL table
df.createOrReplaceTempView("jobs")

In [ ]:
# Verify the Data

# Display the first five rows
df.show(5)

# Show the schema (column names & data types)
df.printSchema()

+--------------------+-----------------+----------------------+----------+----------+----------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+--------------+-------------------+--------------+-------------------+---------------+--------------------+--------------------+--------------------+-------------+-------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-------+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+----

In [ ]:
#How many job postings we have in the dataset?
total_jobs = spark.sql("""
    SELECT COUNT(*) AS total_postings
    FROM jobs
""")
total_jobs.show()

+--------------+
|total_postings|
+--------------+
|         72476|
+--------------+



We have 72,476 total postings.

In [ ]:
#Find the top 5 most common job titles
top_titles = spark.sql("""
    SELECT 
        TITLE_CLEAN AS job_title,
        COUNT(*) AS job_count
    FROM jobs
    WHERE TITLE_CLEAN IS NOT NULL
      AND LOWER(TITLE_CLEAN) != 'unclassified'
    GROUP BY TITLE_CLEAN
    ORDER BY job_count DESC
    LIMIT 5
""")

top_titles.show(truncate=False)

+-----------------------------+---------+
|job_title                    |job_count|
+-----------------------------+---------+
|data analyst                 |4669     |
|enterprise architect         |851      |
|senior data analyst          |786      |
|business intelligence analyst|746      |
|data modeler                 |305      |
+-----------------------------+---------+



The most common role is Data Analyst with 4,669 postings, showing strong demand for analytics skills. Other titles like Enterprise Architect and Data Modeler appear less often, suggesting they are more specialized roles.

In [ ]:
#Find the average salary for each employment type
top_titles = spark.sql("""
    SELECT 
        EMPLOYMENT_TYPE_NAME,
        ROUND(AVG(SALARY),2) AS avg_salary
    FROM jobs
    GROUP BY EMPLOYMENT_TYPE_NAME
""")
top_titles.show()

+--------------------+----------+
|EMPLOYMENT_TYPE_NAME|avg_salary|
+--------------------+----------+
|Part-time / full-...| 105679.79|
|Part-time (≤ 32 h...|  98802.51|
|Full-time (> 32 h...| 118898.35|
+--------------------+----------+



Full-time employment offers both the highest pay and the largest number of opportunities in this dataset — consistent with real-world job market patterns

In [ ]:
#What five states have the most job postings
top_titles = spark.sql("""
    SELECT STATE_NAME, COUNT(*) AS job_count
    FROM jobs
    GROUP BY STATE_NAME
    ORDER BY job_count DESC
    LIMIT 5
""")
top_titles.show(truncate=False)


+----------+---------+
|STATE_NAME|job_count|
+----------+---------+
|Texas     |8067     |
|California|7087     |
|Florida   |3645     |
|Virginia  |3636     |
|Illinois  |3539     |
+----------+---------+



Employment opportunities are concentrated in large and economically diverse states like Texas and California.

In [35]:
#Calculate the salary range (max-min) for each job title in a California
salary_range_ca = spark.sql("""
    SELECT 
        TITLE_NAME,
        MAX(SALARY_TO) AS max_salary,
        MIN(SALARY_FROM) AS min_salary,
        ROUND(MAX(SALARY) - MIN(SALARY),2) AS salary_range,
        COUNT(*) AS job_count
    FROM jobs
    WHERE 
        UPPER(STATE_NAME) LIKE '%CALIFORNIA%'
        AND TITLE_NAME IS NOT NULL
        AND TITLE_NAME !='Unclassified'
        AND SALARY IS NOT NULL
    GROUP BY TITLE_NAME
    ORDER BY salary_range DESC
    LIMIT 5
""")
salary_range_ca.show(truncate=False)


+-------------------------------+----------+----------+------------+---------+
|TITLE_NAME                     |max_salary|min_salary|salary_range|job_count|
+-------------------------------+----------+----------+------------+---------+
|Enterprise Architects          |318600.0  |10230.0   |251142.0    |46       |
|IT Data Analytics Analysts     |384200.0  |48400.0   |241700.0    |22       |
|Enterprise Network Architects  |379500.0  |10893.0   |226850.0    |3        |
|Enterprise Solutions Architects|312200.0  |59800.0   |214250.0    |54       |
|Principal Architects           |327500.0  |12009.0   |203214.0    |30       |
+-------------------------------+----------+----------+------------+---------+



In the job market of California, construction-related positions dominate in terms of salary disparity - the salary range for enterprise architects and chief architects exceeds $200,000 - which reflects significant differences in their qualifications, professional fields, and the scale of the organizations they work for.

In [28]:
#What top 5 industries have the highest average salaries, and ,more than 100 job postings?
top_industries = spark.sql("""
    SELECT 
        LIGHTCAST_SECTORS_NAME AS industry,
        ROUND(AVG(SALARY_TO), 2) AS avg_max_salary,
        ROUND(AVG(SALARY_FROM), 2) AS avg_min_salary,
        ROUND(AVG((SALARY_TO + SALARY_FROM) / 2), 2) AS avg_salary,
        COUNT(*) AS job_count
    FROM jobs
    WHERE 
        LIGHTCAST_SECTORS_NAME IS NOT NULL
        AND SALARY_FROM IS NOT NULL
        AND SALARY_TO IS NOT NULL
    GROUP BY LIGHTCAST_SECTORS_NAME
    HAVING job_count > 100
    ORDER BY avg_salary DESC
    LIMIT 5
""")

top_industries.show(truncate=False)

+---------------------------------------------------------------+--------------+--------------+----------+---------+
|industry                                                       |avg_max_salary|avg_min_salary|avg_salary|job_count|
+---------------------------------------------------------------+--------------+--------------+----------+---------+
|[\n  "Cybersecurity",\n  "Artificial Intelligence"\n]          |172411.43     |108179.8      |140295.62 |368      |
|[\n  "Cybersecurity",\n  "Data Privacy/Protection"\n]          |158428.91     |113769.45     |136099.18 |183      |
|[\n  "Data Privacy/Protection",\n  "Artificial Intelligence"\n]|154997.16     |105271.27     |130134.22 |649      |
|[\n  "Cybersecurity"\n]                                        |149793.93     |105936.68     |127865.3  |1093     |
|[\n  "Artificial Intelligence"\n]                              |155859.3      |99719.17      |127789.24 |3156     |
+---------------------------------------------------------------

Industries combining Cybersecurity and Artificial Intelligence offer the highest average salaries (≈ $140K), followed closely by roles in Data Privacy and standalone AI fields — highlighting that data security and intelligent systems are the most lucrative and in-demand domains.